In [ ]:
import pandas as pd

# Cargar los datasets
df_orders = pd.read_csv('../data/olist_orders_dataset.csv')
df_customers = pd.read_csv('../data/olist_customers_dataset.csv')

# Validamos la carga
display(df_orders.head())

# Visualizamos ahora la cantidad de filas y columnas
cant_filas_columnas_orders = df_orders.shape
cant_filas_columnas_customers = df_customers.shape
print(cant_filas_columnas_customers, cant_filas_columnas_orders)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


(99441, 5) (99441, 8)


In [ ]:
# Diagnóstico de valores nulos (Cuenta cuántos vacíos hay por columna)
print("--- Valores Nulos en la tabla Orders ---")
print(df_orders.isnull().sum())

# Diagnóstico de estructura y tipos de datos
print("\n--- Estructura de la tabla Orders ---")
df_orders.info()

--- Valores Nulos en la tabla Orders ---
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

--- Estructura de la tabla Orders ---
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 

In [ ]:

# Manera más eficiente para todas las columnas, con un bucle for
# Lista con los nombres exactos de todas las columnas que son fechas
columnas_fechas = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

# El ciclo for aplica la transformación de manera eficiente y el coerce aplica NaT a aquellos registros que no pueda transformar Pandas a fechas
for col in columnas_fechas:
    df_orders[col] = pd.to_datetime(df_orders[col], errors='coerce')

# Verificamos que todas hayan cambiado
df_orders.info()

<class 'pandas.Series'>
RangeIndex: 99441 entries, 0 to 99440
Series name: order_purchase_timestamp
Non-Null Count  Dtype         
--------------  -----         
99441 non-null  datetime64[us]
dtypes: datetime64[us](1)
memory usage: 777.0 KB
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  dateti

In [ ]:
# Filtramos el DataFrame para quedarnos SOLO con las filas donde la fecha de entrega es nula
pedidos_sin_entregar = df_orders[df_orders['order_delivered_customer_date'].isnull()]

# De ese grupo filtrado, contamos qué estados de pedido (order_status) tienen
print("--- Estados de los paquetes sin fecha de entrega ---")
print(pedidos_sin_entregar['order_status'].value_counts())
conteo_total_pedidos_sin_entregar = pedidos_sin_entregar['order_status'].value_counts().sum()
print(f'El total es de {conteo_total_pedidos_sin_entregar}')

--- Estados de los paquetes sin fecha de entrega ---
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64
El total es de 2965


In [ ]:
# Filtramos solamente en donde el estado del pedido sea igual a entregado. Reseteamos index para visualizar en formato Data Frame, y drop false para evitar la creación de una columna numérica que se genera por defecto
df_entregados = df_orders[df_orders['order_status'] == 'delivered'].reset_index(drop=True)

total_entregados = len(df_entregados)
display(df_entregados)
print(f'El total de pedidos entregados es de {total_entregados}')

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-02-10 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26
...,...,...,...,...,...,...,...,...
96473,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-09-03 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28
96474,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-06-02 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02
96475,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27
96476,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-08-01 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15


El total de pedidos entregados es de 96478


In [ ]:
# Aplicamos la creación de la columna tiempo de entrega. Para ello usamos la diferencia de la fecha de pedido entrega por la del pedido de compra. Para obtener solo los días, utilizamos .dt.days al final de nuestra operación
df_entregados['tiempo_de_entrega'] = (df_entregados['order_delivered_customer_date'] - df_entregados['order_purchase_timestamp']).dt.days

display(df_entregados)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,tiempo_de_entrega
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-02-10 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,242.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0
...,...,...,...,...,...,...,...,...,...
96473,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-09-03 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,-170.0
96474,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-06-02 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,-94.0
96475,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,24.0
96476,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-08-01 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,-188.0


In [ ]:
# Estadísticamente y visualmente se percibe raro días en negativos, entonces filtramos por la columna en la que se encuentran y traemos solo los positivos. Se pueden descartar los negativos porque son errores del sistema en la carga o en el registro de los datos (ej: se enviaron antes de que se realizara el pedido)
df_filtro_dias_positivos = df_entregados[df_entregados['tiempo_de_entrega'] >= 0].reset_index(drop=True)
display(df_filtro_dias_positivos)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,tiempo_de_entrega
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-02-10 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,242.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0
...,...,...,...,...,...,...,...,...,...
77657,cfa78b997e329a5295b4ee6972c02979,a2f7428f0cafbc8e59f20e1444b67315,delivered,2017-12-20 09:52:41,2017-12-20 10:09:52,2017-12-20 20:25:25,2018-01-26 15:45:14,2018-01-18,37.0
77658,9115830be804184b91f5c00f6f49f92d,da2124f134f5dfbce9d06f29bdb6c308,delivered,2017-04-10 19:57:37,2017-10-04 20:07:14,2017-10-05 16:52:52,2017-10-20 20:25:45,2017-11-07,193.0
77659,aa04ef5214580b06b10e2a378300db44,f01a6bfcc730456317e4081fe0c9940e,delivered,2017-01-27 00:30:03,2017-01-27 01:05:25,2017-01-30 11:40:16,2017-02-07 13:15:25,2017-03-17,11.0
77660,880675dff2150932f1601e1c07eadeeb,47cd45a6ac7b9fb16537df2ccffeb5ac,delivered,2017-02-23 09:05:12,2017-02-23 09:15:11,2017-03-01 10:22:52,2017-03-06 11:08:08,2017-03-22,11.0


In [ ]:
# Exportamos una vez limpios los datos en la tabla para trabajar en SSMS
from sqlalchemy import create_engine

# 1. Credenciales del servidor SQL
nombre_servidor = r'DESKTOP-71D1L52\SQLEXPRESS'
base_datos = 'Olist_Ecommerce'

# 2. Creamos el motor de conexión
string_conexion = f'mssql+pyodbc://@{nombre_servidor}/{base_datos}?driver=ODBC Driver 17 for SQL Server&trusted_connection=yes'
engine = create_engine(string_conexion)


try:
    print("Conectando a SQL Server...")
    # Ejecutamos la consulta y guardamos en Pandas
    df_filtro_dias_positivos.to_sql('fact_pedidos', engine, if_exists='replace', index=False)
    print("¡Conexión exitosa! 🎉\n")
    
except Exception as e:
    print("Error de conexión:")
    print(e)

Conectando a SQL Server...
¡Conexión exitosa! 🎉



In [ ]:
# Repetimos proceso que con df_orders
display(df_customers.head())
df_customers.info()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


In [ ]:
# Contamos todos los valores que sean nulos en la tabla y los sumamos para obtener el total por columna. No se observa ninguno, no hay problema
display(df_customers.isnull().sum())

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [ ]:
# Observamos no tener valores duplicados para la Primary Key del Data Frame clientes, porque la clave primaria es única.
valores_duplicados = df_customers['customer_id'].duplicated().sum()
display(valores_duplicados)

# No realizaremos operaciones matemáticas con customer_zip_code_prefix, así que para evitar conflictos lo pasamos a string
df_customers['customer_zip_code_prefix'] = df_customers['customer_zip_code_prefix'].astype(str)

df_customers.info()

np.int64(0)

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  str  
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: str(5)
memory usage: 3.8 MB


In [ ]:
# Credenciales del servidor SQL
nombre_servidor = r'DESKTOP-71D1L52\SQLEXPRESS'
base_datos = 'Olist_Ecommerce'

# Creamos el motor de conexión
string_conexion = f'mssql+pyodbc://@{nombre_servidor}/{base_datos}?driver=ODBC Driver 17 for SQL Server&trusted_connection=yes'
engine = create_engine(string_conexion)


try:
    print("Conectando a SQL Server...")
    # Ejecutamos la consulta y guardamos en Pandas
    df_customers.to_sql('dim_clientes', engine, if_exists='replace', index=False)
    print("¡Conexión exitosa! 🎉\n")
    
except Exception as e:
    print("Error de conexión:")
    print(e)

Conectando a SQL Server...
¡Conexión exitosa! 🎉

